# Work in progress

In [ ]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
!pip install yellowbrick

# Importing libaries

In [ ]:
import numpy as np
import pandas as pd 


from datetime import datetime
from scipy.stats import skew  # for some statistics
from scipy.special import boxcox1p
from scipy.stats import boxcox_normmax
from sklearn.linear_model import ElasticNetCV, LassoCV, RidgeCV
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import mean_squared_error
from mlxtend.regressor import StackingCVRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import matplotlib.pyplot as plt
import scipy.stats as stats
import sklearn.linear_model as linear_model
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

In [ ]:
train_file_path = "/kaggle/input/house-prices-advanced-regression-techniques/train.csv"
test_file_path = "/kaggle/input/house-prices-advanced-regression-techniques/test.csv"

df_train = pd.read_csv(train_file_path)
df_train_origin = df_train.copy()


df_test = pd.read_csv(test_file_path)
df_test_origin = df_test.copy()

In [ ]:
mask1  = 0<df_test_origin.isna().sum()
mask2 = 0<df_train_origin.isna().sum()
mask1 = set(df_test_origin.columns[mask1])
mask2 = set(df_train_origin.columns[mask2])

# prepocesing

In [ ]:
df_train.drop(columns=["Id"],inplace=True)

In [ ]:
df_train["SalePrice"].max()

Allmost cateogircal columns in df_train is encoded as dtype object. A better suitable dtype is 'cateory'. We will convert those categorical columns to dtype to 'category'.

In [ ]:
from pandas.api.types import is_object_dtype
from pandas.api.types import is_numeric_dtype

numerical_columns = [col for col in df_train.columns if  is_numeric_dtype( df_train[col].dtype) ]
categorical_columns = [col  for col in df_train.columns if is_object_dtype( df_train[col].dtype)]

categorical_columns_with_numeric_dtype = ["OverallQual", "OverallCond","MSSubClass"]
categorical_columns +=categorical_columns_with_numeric_dtype
numerical_columns = [col 
                     for col in numerical_columns 
                         if col not in categorical_columns_with_numeric_dtype
                    ]


print(f"numerical columns are {numerical_columns}")
print(f"categorical columns are {categorical_columns}")

In [ ]:

def convert_cat_value_to_str(df):
    for col in categorical_columns: 
        df[col]= df[col].apply(lambda x:  str(x) if x is not np.nan else x ).astype("category" )
        df[col]= df[col].apply(lambda x:  str(x) if x is not np.nan else x ).astype("category" )

convert_cat_value_to_str(df_train)
convert_cat_value_to_str(df_test)

Some cateorical columns have ordered types. Lets preserve the order. 

In [ ]:
cateogircal_columns_with_order = {
    "OverallQual":dict(map(lambda x:(str(x),x),[1,2,3,4,5,6,7,8,9,10])),
    "OverallCond":dict(map(lambda x:(str(x),x),[1,2,3,4,5,6,7,8,9,10])),
    "ExterQual":{"Po":1,"Fa":2,"TA":3,"Gd":4,"Ex":5},
    "ExterCond":{"Po":1,"Fa":2,"TA":3,"Gd":4,"Ex":5},
    "BsmtQual":{"Po":1,"Fa":2,"TA":3,"Gd":4,"Ex":5},
    "BsmtCond":{"Po":1,"Fa":2,"TA":3,"Gd":4,"Ex":5},
    "BsmtExposure":{"No":1,"Mn":2,"Av":3,"Gd":4},
    "BsmtFinType1":{"Unf":1,"LwQ":2,"Rec":3,"BLQ":4,"ALQ":5,"GLQ":6},
    "BsmtFinType2":{"Unf":1,"LwQ":2,"Rec":3,"BLQ":4,"ALQ":5,"GLQ":6},
    "HeatingQC":{"Po":1,"Fa":2,"TA":3,"Gd":4,"Ex":5},
    "KitchenQual":{"Po":1,"Fa":2,"TA":3,"Gd":4,"Ex":5},
    "FireplaceQu":{"Po":1,"Fa":2,"TA":3,"Gd":4,"Ex":5},
    "GarageQual":{"Po":1,"Fa":2,"TA":3,"Gd":4,"Ex":5},
    "GarageCond":{"Po":1,"Fa":2,"TA":3,"Gd":4,"Ex":5},
    "GarageFinish":{"Unf":1,"RFn":2, "Fin":3},
    "PoolQC":{"Fa":1,"TA":2,"Gd":3,"Ex":4}
}

In [ ]:
cateogircal_columns_with_order.keys()

In [ ]:

def convert_categories(df):
    for col,new_categories in cateogircal_columns_with_order.items():
        df[col] = df[col].cat.rename_categories(new_categories) 
        df[col] = df[col].cat.set_categories(new_categories.values(),ordered=True)
    

    
convert_categories(df_train)   
convert_categories(df_test)   

# Imputation

Some categorical columns use NAN to repsent a missing feature such as garage, basement or pool.

In [ ]:
columns_with_order = {
    "columns": [
    "BsmtQual",
    "BsmtCond",
    "BsmtExposure",
    "BsmtFinType1",
    "BsmtFinType2",
    "FireplaceQu",
    "GarageQual",
    "GarageCond",
    "PoolQC",
    "GarageFinish"
    ],
"nan_value": 0
}

columns_without_order = {
    "columns":[
    "MiscFeature",
    "Fence",
    "Alley",
    "GarageType"
     ],
    "nan_value": "Without"
   }


def modify_nan(df):
    for col in columns_with_order["columns"]:
        value = columns_with_order["nan_value"]
        df[col] = df[col].cat.add_categories(value)
        mask = df[col].isna()
        df[col][mask] = value

    for col in columns_without_order["columns"]:
        value = columns_without_order["nan_value"]
        df[col] = df[col].cat.add_categories(value)
        mask = df[col].isna()
        df[col][mask] = value

modify_nan(df_train)
modify_nan(df_test)

In [ ]:
def calc_prcentege_missing(df):
    na_values_table = (df.isna().sum()/df.shape[0]).sort_values(ascending=False) 
    na_values_table = na_values_table[na_values_table>0] * 100
    return na_values_table

na_values_table_train = calc_prcentege_missing(df_train)
na_values_table_test = calc_prcentege_missing(df_test)

In [ ]:
sns.set_style(style="darkgrid")

plt.figure()
na_values_table_train.plot.bar(title="% NA in train", xlabel="columns" ,ylabel="%")
print(f"There are {na_values_table_train.shape[0]} columns with at least one NA values.")

plt.figure()
na_values_table_test.plot.bar(title="% NA in test", xlabel="columns" ,ylabel="%")
print(f"There are {na_values_table_train.shape[0]} columns with at least one NA values.")




## Imputing train dataset

In [ ]:
is_GarageYrBlt_NANs_due_no_garage = (df_train["GarageYrBlt"].isna() == (df_train["GarageType"] == "Without")).all()


if is_GarageYrBlt_NANs_due_no_garage:
    df_train["GarageYrBlt"].fillna(0,inplace=True)
    print("GarageYrBlt is due to house not having Garage")

In [ ]:
is_related_rows = (df_train["MasVnrArea"].isna() == df_train["MasVnrType"].isna()).all()

if is_related_rows:
    print("the entries with MasVnrArea nan also have MasVnrType as nan")

In [ ]:
fig, (ax1,ax2) =plt.subplots(1,2)
df_train["MasVnrType"].astype("string").fillna("NAN").value_counts().plot.bar(ax=ax1)
df_train["MasVnrArea"].fillna(-1).plot.hist(ax=ax2)

In [ ]:
df_train["MasVnrType"].fillna("None",inplace=True)
df_train["MasVnrArea"].fillna(0,inplace=True)

In [ ]:
df_train["Electrical"].astype("string").fillna("NAN").value_counts().plot.bar()

In [ ]:
most_frequent = df_train["Electrical"].mode().iloc[0]
df_train["Electrical"].fillna(most_frequent,inplace=True)

In [ ]:
fig, (ax1,ax2,ax3) =plt.subplots(1,3, figsize=(15,5))
df_train["LotFrontage"].plot.kde(ax=ax1,title="Frontage")
df_train["LotArea"].plot.kde(ax=ax2,title="Area")
df_train["LotConfig"].value_counts().plot.bar(ax=ax3, title="Config")

In [ ]:
mean = df_train["LotFrontage"].mean()
df_train["LotFrontage"].fillna(mean,inplace=True)

# lets also impute the test samples

In [ ]:
nbr_of_nans = df_test.isna().sum()
nbr_of_nans[nbr_of_nans > 0]

In [ ]:
mask = df_test["GarageYrBlt"].isna() | df_test["GarageArea"].isna()

house_without_garage = (df_test["GarageType"][mask] == "Without") == True
invalid_nans_garage = (df_test["GarageType"][mask] == "Without") == False


invalid_index = df_test[["GarageType","GarageYrBlt"]][mask][invalid_nans_garage].index

print("house with invalid GarageType")
print(invalid_index)

df_test.drop(invalid_index, inplace=True)
df_test["GarageYrBlt"].fillna(0,inplace=True)


In [ ]:
mean = df_test["LotFrontage"].mean()
df_test["LotFrontage"].fillna(mean,inplace=True)

In [ ]:
df_test["MasVnrType"].fillna("None",inplace=True)
df_test["MasVnrArea"].fillna(0,inplace=True)

In [ ]:
df_test = df_test.dropna()

# Outlier dection & EDA

In [ ]:


all_columns_with_order =  columns_with_order["columns"] +  ["OverallQual", "OverallCond"]
all_columns_without_order = columns_without_order["columns"] + ["MSRNType"]

df_ordinal = df_train.select_dtypes("category")[all_columns_with_order]
df_numeric = df_train[numerical_columns]            

corr_matrix = pd.concat([df_ordinal,df_numeric], axis="columns").corr(numeric_only=False)
sns.heatmap(corr_matrix)

In [ ]:
mask = (0.3 < np.abs(corr_matrix["SalePrice"])) & (np.abs(corr_matrix["SalePrice"]) < 1)
correlated_vars = corr_matrix["SalePrice"][mask].sort_values(ascending=False)

In [ ]:
correlated_vars

In [ ]:
from math import ceil
plots_per_row = 3
nbr_of_plots = len(correlated_vars)

fig,axs = plt.subplots( ceil(nbr_of_plots/plots_per_row) ,plots_per_row,figsize=(10,20),sharey=True)
for idx,row in enumerate(axs):
    for idx2,ax in enumerate(row):
        col_index = idx2 +idx*plots_per_row
        if col_index < len(correlated_vars):
            col_name = correlated_vars.index[col_index]
            title = f"{col_name} corr={correlated_vars[col_index]:0.2f}"
            sns.scatterplot(data=df_train, x=correlated_vars.index[col_index],y="SalePrice", ax=ax).set(title=title)

plt.tight_layout()

Looking plots we see that 
 - As excpted OverallQual is most correlated feature to SalePrice.
 - GRLivArea,1stFlrSF and GarageArea have linear relationship to SalePrice.
 - There is clear variance in saleprice boundary between diffrent GarageCars values. The same could be said for  GarageFinsh.  
 - Some of the linear relationship have more spread the larger its feature value is. For example 1stFlrSF and GRLivArea.
 - some of features have clear relationship but its relationship is obscured by a overly dominent value. For example 2ndFloor.

From the plots we see that TotalBsmtSF, GrLivArea and 1stFlrSF have clear linear relationship to SalePrice.

Looking at the plot we see there are 2 houses with overall quality of 10 and low sale price which seems to be outliers.

Looking at the plot we see there are 2 houses with overall quality of 10 and very high sale price which seems to be outliers.
Looking at the year 

In [ ]:
fig, ((ax1,ax2),(ax3,ax4)) =  plt.subplots(2,2,figsize=(15,10))

df = df_train[df_train["2ndFlrSF"]>0]
sns.regplot(data=df_train, x="2ndFlrSF",y="SalePrice",robust=True, ax=ax1,line_kws={"color":"red"})
sns.residplot(data=df_train, x="2ndFlrSF",y="SalePrice",robust=True, ax=ax2)

sns.regplot(data=df, x="2ndFlrSF",y="SalePrice",robust=True, ax=ax3,line_kws={"color":"red"})
sns.residplot(data=df, x="2ndFlrSF",y="SalePrice",robust=True, ax=ax4)
plt.tight_layout()

print("correlation between SalePrice and 2ndFlrSF with houses with 2 floors:",df.corr(numeric_only=True)["SalePrice"]["2ndFlrSF"])
print("correlation between SalePrice and 2ndFlrSF with houses with 2 or 1 floors:",df_train.corr(numeric_only=True)["SalePrice"]["2ndFlrSF"])
print("correlation between SalePrice and GrLivArea:",df_train.corr(numeric_only=True)["SalePrice"]["GrLivArea"])

2ndFlrSF have higher correlation with SalePrice if we asumme there is second floor (eg. 2ndFlrSF > 0).

In [ ]:
fig, (ax1,ax2,ax3) = plt.subplots(3,1,figsize=(15,15))

sns.regplot(data=df_train, x="YearBuilt",y="SalePrice",order=2,line_kws={"color":"red"},ax=ax1)

sns.regplot(data=df_train, x="YearBuilt",y="SalePrice",logx=True,line_kws={"color":"blue"},ax=ax2)

sns.regplot(data=df_train, x="YearBuilt",y="SalePrice",line_kws={"color":"green"},ax=ax3)

A house that is built more recently tends be more expensive. The relationship between YearBuilt and SalePrice is cleary not a simple linear regression. 

A polynom regression of order 2 is better option. We can see that the residuals is more evenly distributed than the simple linear regression.

In [ ]:
from scipy.stats import f_oneway
from scipy.stats import kruskal
from scipy.stats import alexandergovern
all_columns_without_order = "MSZoning","Street","Alley","LotShape","LandContour","Utilities","LotConfig","LandSlope","Neighborhood","Condition1","Condition2","BldgType","HouseStyle","RoofStyle"
"RoofMatl","Exterior1st","Exterior2nd","MasVnrType","Foundation","BsmtFinType2","BsmtFinSF2"
"BsmtUnfSF","Heating","CentralAir","Electrical","GarageType","GarageFinish","PavedDrive","Fence","SaleType","SaleCondition"

nominal_with_diffrent_median = []
for nominal in all_columns_without_order:
    p_value = kruskal(*df_train.groupby(nominal)["SalePrice"].apply(list)).pvalue
    #a = alexandergovern(*df_train.groupby(nominal)["SalePrice"].apply(list)).pvalue
    
    if p_value < 0.05:
        nominal_with_diffrent_median.append(nominal)
    print(f"{nominal}:{p_value:0.2f}")

In [ ]:
len(nominal_with_diffrent_median)

In [ ]:
df_train["MSZoning"].value_counts()

In [ ]:
fig, axs = plt.subplots(4,3,figsize=(20,20))
axs = axs.reshape((-1,1)).squeeze()
for ax,col in zip(axs,nominal_with_diffrent_median):
    #sns.stripplot(df_train, x=col, y="SalePrice", ax=ax)
    my_plot = sns.boxplot(df_train, x=col, y="SalePrice", ax=ax)
    my_plot.set_xticklabels(my_plot.get_xticklabels(), rotation=45)
plt.tight_layout()

From the plots we can see 
- A House in low residential density (MSZoning RL) is can be more expensive than house in high or medium residential density (MSZoning RH or RM).
- In general a house with a paved alley is more expensive than graveled alley.
- Neihboorhood have clear impact on Price.

Let to try find unusual sale prices. Will later check those house with other properties to see if there is clear reason for the unusual sale prices.

In [ ]:
# contains index of outliers and the method used for decting
outliers = dict()

In [ ]:
# IQR 
fig,(ax1,ax2) = plt.subplots(1,2,figsize=(15,5))
sns.boxplot(df_train["SalePrice"], orient="h", ax=ax1)
sns.histplot(df_train["SalePrice"],stat="probability",kde=True, ax=ax2)


Q1 = df_train["SalePrice"].quantile(0.25)
Q3 = df_train["SalePrice"].quantile(0.75)
IQR = Q3-Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR


mask1 = (df_train["SalePrice"] < lower) | ( upper < df_train["SalePrice"] ) 
df = df_train.copy()
df["outliers"] = mask1

outliers["IQR"] = df_train[mask1].index.to_list()

df_train.loc[~mask1]
fig,(ax1,ax2) = plt.subplots(1,2,figsize=(15,5))
sns.stripplot(data=df, x="SalePrice",orient="h",hue="outliers", ax=ax1)
sns.histplot(df_train["SalePrice"][~mask1],stat="probability",kde=True, ax=ax2)



#corr_after = df_train.loc[~mask1].corr(numeric_only=True)["SalePrice"].sort_values(ascending=False)
#diffrance_in_index = corr_before.index == corr_after.index
#x = pd.DataFrame()
#x["before"] = corr_before.index
#x["after"] = corr_after.index 
#x["diffrent"] = corr_before.index == corr_after.index
#x["before_value"] = corr_before.values
#x["after_value"] = corr_after.values

#x.loc[diffrance_in_index]

There seems to many outlies using IQR. The outlier are proably because for its right tail. However there is only two point there is very far off.
Let check with other methods.

In [ ]:
# Percentile Capping 
below = np.percentile(df_train["SalePrice"] , 1)
above = np.percentile(df_train["SalePrice"] , 99)

mask2 = (df_train["SalePrice"] < below) | ( above < df_train["SalePrice"] ) 
outliers["Percentile"] = df_train[mask2].index.to_list()
df = df_train.copy()

fig,(ax1,ax2) = plt.subplots(1,2,figsize=(15,5))
fig.suptitle('Outliers by Percentile capping', fontsize=16)
df["outliers"] = mask2
ax = sns.stripplot(data=df, x="SalePrice",orient="h",hue="outliers", ax=ax1)
sns.histplot(df_train["SalePrice"][~mask2],stat="probability",kde=True, ax=ax2)
#ax.axvline(x = df_train["SalePrice"].mean(), color="red")
#ax.axvline(x = df_train["SalePrice"].median(), color="green")


#corr_before = df_train.corr(numeric_only=True)["SalePrice"].sort_values(ascending=False)
#corr_after = df_train.loc[~mask2].corr(numeric_only=True)["SalePrice"].sort_values(ascending=False)
#diffrance_in_index = corr_before.index != corr_after.index
#x = pd.DataFrame()
#x["before"] = corr_before.index
#x["after"] = corr_after.index 
#x["diffrent"] = corr_before.index != corr_after.index
#x["before_value"] = corr_before.values
#x["after_value"] = corr_after.values
#x.loc[diffrance_in_index]

In [ ]:
#fig,axs = plt.subplots(5,5,figsize=(10,15))
#axs  = axs.reshape((-1,1)).squeeze()
#
#
#for col,ax in zip(numerical_columns,axs):
#    sns.boxplot(df_train, x=col, ax=ax)
#    #sns.histplot(df_train["SalePrice"],stat="probability",kde=True, ax=ax2)
#plt.tight_layout()
#fig,(ax1,ax2) = plt.subplots(1,2,figsize=(15,5))
#sns.stripplot(data=df, x="SalePrice",orient="h",hue="outliers", ax=ax1)
#sns.histplot(df_train["SalePrice"][~mask1],stat="probability",kde=True, ax=ax2)

Percentile capping is less senestive to the far points on the right but considers instead some of point on the left as outliers.

In [ ]:
# z-score
# From previous observation we know sale price is not normal distributed.
# lets transform it with log and use zscore  with points outside 3 * sigma as outlier boundary


from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import power_transform
fig,(ax1,ax2,ax3) = plt.subplots(1,3,figsize=(15,5))
fig.suptitle('Outliers by z-score 3 standred deviation as boundary', fontsize=16)



orginal_saleprice_dist = StandardScaler().fit_transform(df_train["SalePrice"].to_numpy().reshape((-1,1)))
sns.histplot(orginal_saleprice_dist,stat="probability",kde=True, ax=ax1).set(title="orignal")


log_saleprice_dist = StandardScaler().fit_transform(np.log(df_train["SalePrice"]).to_numpy().reshape((-1,1)))
sns.histplot(log_saleprice_dist,stat="probability",kde=True, ax=ax2).set(title="log transformation")

x = pd.DataFrame(columns=["orignal","log"])


print("(Excess) kurtosis for orginal SalePrice",stats.kurtosis(orginal_saleprice_dist))
print("(Excess) kurtosis for log transformed SalePrice",stats.kurtosis(log_saleprice_dist))

if  0 < stats.kurtosis(log_saleprice_dist):
    print("transformed Saleprice is leptokurtic (fat tails)")

elif stats.kurtosis(log_saleprice_dist) < 0:
    print("transformed Saleprice is platykurtic (small tails)")

print("skewness for orginal SalePrice",stats.skew(orginal_saleprice_dist) )
print("skewness for log transformed SalePrice",stats.skew(log_saleprice_dist) )

#sns.histplot(np.log(df_train["SalePrice"]),stat="probability",kde=True, ax=ax2, title="log")



mask3 = 3 < np.abs((log_saleprice_dist - log_saleprice_dist.mean()/log_saleprice_dist.std())) 
outliers["Zscore"] = df_train[mask3].index.to_list()
df = df_train.copy()
df["outliers"] = mask3


sns.stripplot(data=df, x="SalePrice",orient="h",hue="outliers",ax=ax3)
plt.tight_layout()

#corr_before = df_train.corr(numeric_only=True)["SalePrice"].sort_values(ascending=False)
#corr_after = df_train.loc[~mask3].corr(numeric_only=True)["SalePrice"].sort_values(ascending=False)
#diffrance_in_index = corr_before.index != corr_after.index
#x = pd.DataFrame()
#x["before"] = corr_before.index
#x["after"] = corr_after.index 
#x["diffrent"] = corr_before.index != corr_after.index
#x["before_value"] = corr_before.values
#x["after_value"] = corr_after.values
#x.loc[diffrance_in_index]


# Outliers By Cutoff levels 

In [ ]:

max_area = 2935
max_bath_room = 10
max_rooms = 20
max_fire_place = max_rooms
max_garage_cars = 10
max_pool_area = 538

cut_off_levels = (
 ('LotFrontage',0,np.inf),
 ('LotArea',0, np.inf),
 ('YearBuilt',0,np.inf),
 ('YearRemodAdd',0,np.inf),
 ('MasVnrArea',0,max_area),
 ('BsmtFinSF1',0,max_area),
 ('BsmtFinSF2',0,max_area),
 ('BsmtUnfSF',0,max_area),
 ('TotalBsmtSF',0,max_area),
 ('1stFlrSF',0,max_area),
 ('2ndFlrSF',0,max_area),
 ('LowQualFinSF',0,max_area),
 ('GrLivArea',0,max_area),
 ('BsmtFullBath',0,max_bath_room),
 ('BsmtHalfBath',0,max_bath_room),
 ('FullBath',1,max_bath_room),
 ('HalfBath',0,max_bath_room),
 ('BedroomAbvGr',1,max_rooms),
 ('KitchenAbvGr',1,max_rooms),
 ('TotRmsAbvGrd',1,max_rooms),
 ('Fireplaces',0,max_fire_place),
 ('GarageYrBlt',0,np.inf),
 ('GarageCars',0,max_garage_cars),
 ('GarageArea',0,max_area),
 ('WoodDeckSF',0,np.inf),
 ('OpenPorchSF',0,np.inf),
 ('EnclosedPorch',0,np.inf),
 ('3SsnPorch',0,np.inf),
 ('ScreenPorch',0,np.inf),
 ('PoolArea',0,max_pool_area),
 ('MoSold',1,12),
 ('YrSold',0,np.inf)
)

data = {
    "min":[min for _,min,_ in cut_off_levels],
    "max":[max for _,_,max in cut_off_levels],
}
df_cut_of = pd.DataFrame(data, index=[ f for f,_,_ in cut_off_levels])

outside_cut_of_level=[]
for f in df_cut_of.index:
    mask = ~(df_cut_of.loc[f]["min"] <= df_train[f]) & (df_train[f] <= df_cut_of.loc[f]["max"])
    if mask.any():
        outside_cut_of_level.append({"f":f,"mask":mask})

outliers["CutOff"] = []
for data in outside_cut_of_level:
    f = data["f"]
    mask = data["mask"]
    print(f)
    print(df_train[f][mask])
    outliers["CutOff"].extend(df_train[f][mask].index.to_list())

# Finding invalid configuration

In [ ]:
outliers["Invalid"] = []
print("invalid configuration")
mask = df_train["GrLivArea"] < df_train["GarageArea"]
if mask.any():
    print(df_train.loc[mask][["GrLivArea","GarageArea","GarageType"]])
    outliers["Invalid"].extend(df_train[mask].index.to_list())

mask = df_train["GrLivArea"] < df_train["1stFlrSF"]
if mask.any():
    print(df_train[["GrLivArea","1stFlrSF"]])
    outliers["Invalid"].extend(df_train[mask].index.to_list())
    
mask = (df_train["PoolArea"] == 0) & (df_train["PoolQC"] > 0)
if mask.any():
    print(df_train[["GrLivArea","1stFlrSF"]])
    outliers["Invalid"].extend(df_train[mask].index.to_list())

mask = (df_train["Fireplaces"] == 0) & (df_train["FireplaceQu"] > 0)
if mask.any():
    print(df_train[["Fireplaces","FireplaceQu"]])
    outliers["Invalid"].extend(df_train[mask].index.to_list())

mask =  df_train["TotRmsAbvGrd"] < df_train["BedroomAbvGr"] 
if mask.any():
    print(df_train[["TotRmsAbvGrd","BedroomAbvGr"]])
    outliers["Invalid"].extend(df_train[mask].index.to_list())

mask = df_train["YearRemodAdd"] < df_train["YearBuilt"]
if mask.any():
    print(df_train[["YearRemodAdd","YearBuilt"]])
    outliers["Invalid"].extend(df_train[mask].index.to_list())

mask =  1 < np.abs(df_train["BsmtFinSF1"] + df_train["BsmtFinSF2"] + df_train["BsmtUnfSF"] -  df_train['TotalBsmtSF']) 
if mask.any():
    print(df_train[["BsmtFinSF1","BsmtFinSF2","BsmtUnfSF",'TotalBsmtSF']])
    outliers["Invalid"].extend(df_train[mask].index.to_list())

Lets remove the all outliers. 

In [ ]:

common_outliers = set(outliers["IQR"]) & set(outliers["Percentile"]) & set(outliers["Zscore"]) & set(outliers["CutOff"]) & set(outliers["Invalid"])
all_outliers = set(outliers["IQR"]) | set(outliers["Percentile"]) | set(outliers["Zscore"]) | set(outliers["CutOff"]) | set(outliers["Invalid"])
print("common outliers",common_outliers)
print("all_outliers",all_outliers)

In [ ]:
df_train_purged = df_train.drop(common_outliers, axis="index")

In [ ]:
def add_features(df):
    df_extend = df.copy()
    df_extend["room_area"] = df["GrLivArea"]/df["TotRmsAbvGrd"]
    df_extend["ratio_bedrooms"] = df["TotRmsAbvGrd"]/(df["BedroomAbvGr"] +1)
    df_extend["total_area"] = df["TotalBsmtSF"] + df["GrLivArea"] 
    df_extend["total_bath_rooms"] = df["BsmtFullBath"] + 0.5*df["BsmtHalfBath"] + df["FullBath"] + 0.5*df["HalfBath"] 
    df_extend["special_rooms"] = df["FullBath"].astype("int32") + df["BedroomAbvGr"].astype("int32") + df["KitchenAbvGr"].astype("int32")
    df_extend["area_space_for_car"] = df["GarageArea"] / (df["GarageCars"] +1)
    return df_extend

df_train_extend = add_features(df_train_purged)
df_test_extend = add_features(df_test)
df_train_extend.corr(numeric_only=True)["SalePrice"].sort_values(ascending=False)

let start creating models.

# Base line model

In [ ]:
df_train_extend.min(numeric_only=True)

In [ ]:
df = pd.DataFrame([[0],[2],[3],[4]], columns=["s"])

df[["s"]].where(df[["s"]] !=0, 1e-7)  

In [ ]:
df_train["SalePrice"]

In [ ]:
from sklearn.preprocessing import PowerTransformer, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn import metrics 
from sklearn_pandas import DataFrameMapper
from pandas.api.types import is_categorical_dtype, is_numeric_dtype
from sklearn.model_selection import train_test_split

def bulid_pipline(X,y):
    with_order = [ key for key,col in X.items() if is_categorical_dtype(col.dtype) and col.cat.ordered is True]
    with_out_order = [ key for key,col in X.items() if is_categorical_dtype(col.dtype) and col.cat.ordered is False]
    numeric = [ key for key,col in X.items() if is_numeric_dtype(col.dtype) ]

    numeric_and_ordinal = numeric + with_order
    

    encoder = OneHotEncoder(categories='auto')
    transformer = PowerTransformer()

    pipeline_x =  ColumnTransformer([
        ("transformer",transformer,numeric_and_ordinal),
        ("encoder",encoder,with_out_order)
    ], remainder="passthrough")
    
    pipeline_y =  PowerTransformer()
    
    return pipeline_x.fit(X), pipeline_y.fit(y)


X,y =df_train.drop(columns=["SalePrice"]), df_train["SalePrice"].to_numpy().reshape(-1,1) 

pipline_X,pipline_y = bulid_pipline(X,y)
X = pipline_X.transform(X)
y = pipline_y.transform(y)

X_train, X_test, y_train, y_test =  train_test_split(X,y, test_size=0.33,random_state=42)

# Lets build base models that predicts prices

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.tree import DecisionTreeRegressor 
from sklearn.linear_model import LinearRegression,Ridge,ElasticNet,Lasso
from sklearn.preprocessing import PolynomialFeatures
from sklearn.ensemble import RandomForestRegressor
from xgboost.sklearn import XGBRegressor
from lightgbm import LGBMRegressor

from sklearn.svm import SVR

from sklearn.model_selection import cross_val_score
from sklearn.metrics import  make_scorer,r2_score
from sklearn.svm import SVR


def test_performance(models,X_train, X_test, y_train, y_test,metrics,fit=True):
    y_train = y_train.reshape(-1)
    y_test = y_test.reshape(-1)
    
    results = []
    for model in models:
        
        model_info = {"model": model.__class__.__name__}
        results.append(model_info)
        
        if fit:
            model.fit(X_train,y_train)
        
        for metric in metrics:
            
            y_pred_train = model.predict(X_train)
            y_pred_test = model.predict(X_test)
            
            model_info["train_" + metric.__name__] = metric(y_pred_train,y_train)
            model_info["test_" + metric.__name__] = metric(y_pred_test,y_test)
    
        
    
    return results

models = [LinearRegression(),Ridge(),ElasticNet(),DecisionTreeRegressor(),SVR(),Lasso(),RandomForestRegressor()]
metrics = [r2_score, mean_absolute_error, mean_squared_error]
data = test_performance(models,X_train, X_test, y_train, y_test,metrics)
pd.DataFrame(data)

# Improving Ridge

In [ ]:
def find_good_apha_ridge(X_train,y_train, rounds = 3):
    model = RidgeCV().fit(X_train,y_train)
    
    for i in range(rounds):
        best_alpha = model.alpha_
        new_alphas = np.logspace(np.log10(best_alpha)/10, np.log10(best_alpha)*10, 10)
        model = RidgeCV(alphas  = new_alphas).fit(X_train,y_train)


    return model

model = find_good_apha_ridge(X_train,y_train, rounds=3)

data = test_performance([model],X_train, X_test, y_train, y_test,metrics,fit=False)

pd.DataFrame(data)

In [ ]:
number_of_zeroed_coef = np.sum(model.coef_ == 0)

print(f"number of zeroed coefficients is {number_of_zeroed_coef}")

# Improving Lasso 

In [ ]:
from sklearn.linear_model import LassoCV
def find_good_alpha_lasso(X_train,y_train, rounds = 3):
    y_train = y_train.reshape(-1)
    
    model = LassoCV().fit(X_train,y_train)
    
    for i in range(rounds):
        best_alpha = model.alpha_
        new_alphas = np.linspace(np.log10(best_alpha)-4, np.log10(best_alpha)+4, 20)
        model.fit(X_train,y_train)


    return model

model = find_good_alpha_lasso(X_train,y_train)

data = test_performance([model],X_train, X_test, y_train, y_test,metrics,fit=False)

pd.DataFrame(data)

In [ ]:
number_of_zeroed_coef = np.sum(model.coef_ == 0)

print(f"number of zeroed coefficients is {number_of_zeroed_coef}")

In [ ]:
from sklearn.linear_model import Lasso
from sklearn.pipeline import Pipeline
def with_poly_features(X_train,y_train, alpha = 3):
    y_train = y_train.reshape(-1)
    model = Pipeline([
        ("feature", PolynomialFeatures(2,include_bias=False )),
        ("lasso",Lasso(alpha = alpha, max_iter=4000))
    ])
    model.fit(X_train,y_train)
    return model


model = with_poly_features(X_train,y_train, alpha = model.alpha_)

data = test_performance([model],X_train, X_test, y_train, y_test,metrics,fit=False)

pd.DataFrame(data)

# Improving Descsion Tree

In [ ]:
from sklearn.decomposition import PCA
from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import GridSearchCV

def reduce_features(X_train,y_train,variance_threshold_drop = 1 - 0.90,variance_threshold_explained=0.90,search_space = None,cv=5):
    if search_space is None:
        search_space = {
         "criterion": ["squared_error", "absolute_error"],
          "min_samples_split": [10, 20, 40,80,120],
          "splitter":["best"],
          "max_depth": [3,4,6],
          "min_samples_leaf": [3, 8,10,12],
        }
    
    estimator_name  = "tree"
    
    search_space = { f"{estimator_name}__{key}":value for key,value in search_space.items()}

    
    v = VarianceThreshold(variance_threshold_drop)
    pca = PCA(n_components=variance_threshold_explained)
    tree = DecisionTreeRegressor()
    
    pipe = Pipeline([
        ("VarianceThreshold",v),
        ("PCA",pca),
        (estimator_name,tree)
    ])
    
    model = GridSearchCV(pipe, search_space, cv=cv)
    model.fit(X_train,y_train)
 
    return model

model = reduce_features(X_train,y_train)

data = test_performance([model],X_train, X_test, y_train, y_test,metrics,fit=False)
print("best parametes:",model.best_params_)
pd.DataFrame(data)

# Improving Linear regression

In [ ]:
from sklearn.feature_selection import SelectorMixin 
from sklearn.base import BaseEstimator
import numpy as np
from numpy.typing import NDArray

class LowCorrelatedFeaturesSelector(SelectorMixin, BaseEstimator):

    def __init__(self, threshold = 0.8):
        self.threshold = threshold


    def __sort_corr_indices_min_to_max(self, corr:NDArray, indices:tuple[NDArray[...], ...]):
        values = corr[indices]
        smallest_to_largest = values.argsort()[::-1]
        reversed_indices = (indices[0][smallest_to_largest],indices[1][smallest_to_largest])
        return reversed_indices



    def __find_high_correlated_features(self,X:NDArray, threshold, sort_max_to_min=True):
        corr = np.corrcoef(X,rowvar=False) 
        ones = np.nonzero( (threshold < np.abs(corr)) & (np.abs(corr) < 1) )

        mask  = np.zeros(shape = corr.shape)
        mask[ones] = 1
        mask[np.tri(N=mask.shape[0]).astype(bool)] = 0
        idxs = np.where(mask == 1)

        if sort_max_to_min:
            idxs = self.__sort_corr_indices_min_to_max(corr, idxs)

        return corr, idxs  


    def __remove_high_correlated_features(self,X:NDArray,y, threshold):
        corr, idxs = self.__find_high_correlated_features(X,threshold=threshold)
        
        corr_with_y:NDArray = np.corrcoef(X,rowvar=False, y = y)[-1,0:-1] 
        corr_with_y = np.abs(corr_with_y)

        feature_selected =  np.ones(shape = corr_with_y.shape) 
        highly_correlated_pairs = list()
        all_highly_correlated_features = set()
        for row,col in zip(idxs[0],idxs[1]):
            highly_correlated_pairs.append(set([row,col]))
            all_highly_correlated_features.add(row)
            all_highly_correlated_features.add(col)
        

        for feature_x in all_highly_correlated_features:
            if feature_selected[feature_x]  == 0:
                continue

            all_correlated_features_with_x = list(filter(lambda t: feature_x in t, highly_correlated_pairs))
            all_correlated_features_with_x = set().union(*all_correlated_features_with_x) - {feature_x}
            
            for j in all_correlated_features_with_x:
                if corr_with_y[j] <= corr_with_y[feature_x]:
                    feature_selected[j] = 0
        
        return feature_selected.astype(np.bool_) #{corr,idxs,corr_with_y,highly_correlated_pairs}
    
    def fit(self,X,y=None):
        self.__masks = self.__remove_high_correlated_features(X,y, threshold= self.threshold)
        return self

    
    def _get_support_mask(self):
        return self.__masks
    
from yellowbrick.regressor import ResidualsPlot

def improve_linear_regression(X_train,y_train):
    model = Pipeline([
        ("variance-selector", VarianceThreshold(0.1)),
        ("corrlation-selector", LowCorrelatedFeaturesSelector(threshold=0.9)),
        ("estimator",LinearRegression()),
    ])
    model.fit(X_train, y_train)  
    return model

model = improve_linear_regression(X_train,y_train)

data = test_performance([model],X_train, X_test, y_train, y_test,metrics,fit=False)
pd.DataFrame(data)

In [ ]:
feature_to_be_used = ["GrLivArea","GarageArea","TotalBsmtSF","YearBuilt","total_Bath","have_Fireplace","have_WoodDeck","have_Porch"]
all_cols = [*feature_to_be_used, "SalePrice"]
df = df_train.copy()

df["total_Bath"] =  df_train["FullBath"] + 0.5 * df_train["HalfBath"] + 0.5 * df_train["BsmtHalfBath"] + df_train["BsmtFullBath"]
df["have_Fireplace"] =  df_train["Fireplaces"] > 0
df["have_WoodDeck"] = df_train["WoodDeckSF"] > 0
df["have_Porch"] = (df_train["OpenPorchSF"] > 0) |( df_train["3SsnPorch"] > 0) | (df_train["ScreenPorch"] > 0) | (df_train["EnclosedPorch"]>0) 



df = df[all_cols] 

def cap_at(df, cap_level  = 1):
    outliers_indices = set()
    
    for key in df.keys():
        below = np.percentile(df["SalePrice"] , cap_level)
        above = np.percentile(df["SalePrice"] , 100 - cap_level)

        mask = (df["SalePrice"] < below) | ( above < df["SalePrice"] ) 
        outliers = df[mask].index.to_list()
        outliers_indices = outliers_indices.union(outliers) 
    
    outliers_indices = list(outliers_indices)
    return df.drop(outliers_indices, axis="index")


df = cap_at(df)
    
transformer_for_X = PowerTransformer()
transformer_for_y = PowerTransformer()
transformers = (transformer_for_X,transformer_for_y)



X = df[feature_to_be_used].to_numpy()
y = df["SalePrice"].to_numpy().reshape(-1,1)


X = transformer_for_X.fit_transform(X)
y = transformer_for_y.fit_transform(y)

X_train, X_test, y_train, y_test =  train_test_split(X,y, test_size=0.33,random_state=42)

model = LinearRegression(positive=True).fit(X_train, y_train)

data = test_performance([model],X_train, X_test, y_train, y_test,metrics,fit=False)
pd.DataFrame(data)


In [ ]:
import pickle
with open("transformers.p","wb") as f1:
    pickle.dump(transformers, f1)
    
model.fit(X,y)

with open("model.p","wb") as f2:
    pickle.dump(model, f2)

defaults = {
    pd.DataFrame.mean:["GrLivArea","GarageArea","TotalBsmtSF","YearBuilt"], 
    pd.DataFrame.mode:["have_Fireplace","have_WoodDeck","have_Porch","total_Bath"] 
}

defaults = pd.concat([
    df[defaults[pd.DataFrame.mean]].mean().round().to_frame().transpose(),
    df[defaults[pd.DataFrame.mode]].mode()
], axis="columns")
defaults = defaults[feature_to_be_used]

defaults.to_csv("defaults.csv")

In [ ]:
def guess(x):
    y = model.predict(x)
    x = transformer_for_X.inverse_transform(x)
    y = transformer_for_y.inverse_transform(y)
    return x,y

x_g,y_g = guess(X[[13]])
data = dict(zip(all_cols,[*x_g[0],y_g[0][0]]))
data = { k:[v] for k,v in data.items()}
data["new_point"] = True 

df_new = pd.DataFrame(data)
dfx = df.copy()
dfx["new_point"] = False 
dfx = pd.concat([dfx,df_new])

In [ ]:
defaults.iloc[0].tolist()

In [ ]:
df[feature_to_be_used].iloc[0]

In [ ]:
X[[0]]

In [ ]:
transformer_for_y.inverse_transform(model.predict(X[[0]]))

In [ ]:
model.predict(X[[0]])

In [ ]:
model.predict(transformer_for_X.transform(df[feature_to_be_used].to_numpy()[[0]]))

In [ ]:
sns.pairplot(
    dfx,
    x_vars=feature_to_be_used,
    y_vars=["SalePrice"],
    hue="new_point"
)